<a href="https://colab.research.google.com/github/Milu2662/BI-PYMES-ML/blob/main/notebook/05_local_outlier_factor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BI-PYMES-ML - LOCAL OUTLINER FACTOR


In [2]:
!git clone https://github.com/Milu2662/BI-PYMES-ML.git /content/BI-PYMES-ML

fatal: destination path '/content/BI-PYMES-ML' already exists and is not an empty directory.


In [3]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_DIR = Path("/content/BI-PYMES-ML")

if not (PROJECT_DIR / "src").exists():
    raise FileNotFoundError("No se encontró el proyecto.")

sys.path.insert(0, str(PROJECT_DIR))

from google.colab import drive
drive.mount("/content/drive")

print("Proyecto:", PROJECT_DIR)

Mounted at /content/drive
Proyecto: /content/BI-PYMES-ML


In [4]:
import shutil

drive_file = Path(
    "/content/drive/MyDrive/BI-PYMES-ML/invoice_features_enriched.pkl"
)

local_file = (
    PROJECT_DIR /
    "data" /
    "processed" /
    "invoice_features_enriched.pkl"
)

local_file.parent.mkdir(parents=True, exist_ok=True)

if not drive_file.exists():
    raise FileNotFoundError(
        "No se encontró invoice_features_enriched.pkl en Drive. "
        "Debes guardarlo desde Notebook 04."
    )

shutil.copy2(drive_file, local_file)

invoice_enriched = pd.read_pickle(local_file)

print("Dimensiones:", invoice_enriched.shape)
print("Facturas:", invoice_enriched["InvoiceNo"].nunique())

display(invoice_enriched.head())

Dimensiones: (53628, 27)
Facturas: 53628


,InvoiceNo,line_count,total_quantity,mean_quantity,mean_unit_price,total_revenue,revenue_abs,min_unit_price,max_unit_price,cancellation_line_share,...,share_BAD_DEBT_ADJUSTMENT,share_BANK_CHARGE,share_CARRIAGE,share_COMMISSION,share_DISCOUNT,share_MANUAL_ADJUSTMENT,share_PLATFORM_FEE,share_POSTAGE,share_SAMPLES,share_STANDARD_PRODUCT
0,489434,8,166,20.750000,4.081250,505.30,505.30,1.25,6.95,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,489435,4,60,15.000000,2.625000,145.80,145.80,1.65,3.75,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,489436,19,193,10.157895,3.730526,630.33,630.33,1.25,8.50,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,489437,23,145,6.304348,3.628261,310.75,310.75,0.65,9.95,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,489438,17,826,48.588235,2.591176,2286.24,2286.24,0.98,6.40,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## 5. Local Outlier Factor y línea base estadística MAD

Se implementan dos métodos complementarios de detección de observaciones atípicas a nivel de factura.

**Local Outlier Factor (LOF):** identifica observaciones cuya densidad local difiere de la de sus vecinos. Se configura con 20 vecinos y una contaminación del 3 %. El modelo se ajusta sobre el conjunto disponible (`novelty=False`), por lo que sus resultados describen detección sobre los datos analizados y no generalización a datos futuros.

**Baseline MAD:** utiliza la desviación absoluta mediana para calcular puntuaciones robustas por característica. Una factura se clasifica como atípica cuando al menos una variable supera el umbral de 3,5.

Ambos métodos utilizan las mismas 14 características para facilitar la comparación de sus resultados.

In [17]:
from src.anomaly_detection import fit_local_outlier_factor

# Características revisadas: 14 variables
lof_features_revised = [
    "line_count",
    "total_quantity",
    "mean_quantity",
    "mean_unit_price",
    "total_revenue",
    "revenue_abs",
    "min_unit_price",
    "max_unit_price",
    "special_line_share",
    "share_POSTAGE",
    "share_MANUAL_ADJUSTMENT",
    "share_DISCOUNT",
    "share_BANK_CHARGE",
    "share_PLATFORM_FEE"
]

# Ejecutar LOF
model_lof, scored_lof, lof_features = fit_local_outlier_factor(
    invoice_enriched,
    contamination=0.03,
    n_neighbors=20,
    feature_cols=lof_features_revised
)

print("LOCAL OUTLIER FACTOR REVISADO")
print("-" * 50)
print("Facturas evaluadas:", len(scored_lof))
print("Variables utilizadas:", len(lof_features))
print("Anomalías:", int(scored_lof["lof_anomaly"].sum()))
print(
    "Porcentaje:",
    round(scored_lof["lof_anomaly"].mean() * 100, 3),
    "%"
)

display(
    scored_lof.sort_values(
        "lof_score",
        ascending=False
    ).head(15)
)

LOCAL OUTLIER FACTOR REVISADO
--------------------------------------------------
Facturas evaluadas: 53628
Variables utilizadas: 14
Anomalías: 1607
Porcentaje: 2.997 %


/usr/local/lib/python3.13/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


,InvoiceNo,line_count,total_quantity,mean_quantity,mean_unit_price,total_revenue,revenue_abs,min_unit_price,max_unit_price,cancellation_line_share,...,share_COMMISSION,share_DISCOUNT,share_MANUAL_ADJUSTMENT,share_PLATFORM_FEE,share_POSTAGE,share_SAMPLES,share_STANDARD_PRODUCT,lof_label,lof_anomaly,lof_score
45330,A506401,1,1,1.0,-53594.360,-53594.36,53594.36,-53594.36,-53594.36,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1,1,3.914701e+14
45331,A516228,1,1,1.0,-44031.790,-44031.79,44031.79,-44031.79,-44031.79,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1,1,3.294157e+14
45332,A528059,1,1,1.0,-38925.870,-38925.87,38925.87,-38925.87,-38925.87,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1,1,2.962818e+14
45334,A563186,1,1,1.0,-11062.060,-11062.06,11062.06,-11062.06,-11062.06,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1,1,1.013437e+14
45335,A563187,1,1,1.0,-11062.060,-11062.06,11062.06,-11062.06,-11062.06,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1,1,1.013437e+14
20511,530609,1,100,100.0,0.000,0.00,0.00,0.00,0.00,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-1,1,1.323828e+10
53254,C576244,1,-1,-1.0,6.000,-6.00,6.00,6.00,6.00,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,-1,1,1.273740e+10
48032,C517262,1,-1,-1.0,6.000,-6.00,6.00,6.00,6.00,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,-1,1,1.273740e+10
50436,C543624,1,-200,-200.0,0.410,-82.00,82.00,0.41,0.41,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-1,1,1.192858e+10
45869,C494909,2,-60,-30.0,0.265,-15.90,15.90,0.13,0.40,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,-1,1,1.144222e+10


In [18]:
from src.anomaly_detection import statistical_baseline

scored_mad, mad_features = statistical_baseline(
    invoice_enriched,
    feature_cols=lof_features_revised,
    threshold=3.5
)

print("BASELINE ESTADÍSTICO MAD")
print("-" * 50)x
print("Facturas evaluadas:", len(scored_mad))
print("Variables utilizadas:", len(mad_features))
print(
    "Anomalías:",
    int(scored_mad["baseline_anomaly"].sum())
)
print(
    "Porcentaje:",
    round(scored_mad["baseline_anomaly"].mean() * 100, 3),
    "%"
)

BASELINE ESTADÍSTICO MAD
--------------------------------------------------
Facturas evaluadas: 53628
Variables utilizadas: 14
Anomalías: 21359
Porcentaje: 39.828 %


In [20]:
from pathlib import Path
import shutil
import joblib

output_dir = PROJECT_DIR / "results" / "metrics"
output_dir.mkdir(parents=True, exist_ok=True)

# Guardar resultados LOF
scored_lof.to_csv(
    output_dir / "local_outlier_factor_scores.csv",
    index=False
)

joblib.dump(
    model_lof,
    output_dir / "local_outlier_factor_model.joblib"
)

# Guardar resultados MAD
scored_mad.to_csv(
    output_dir / "statistical_baseline_mad_scores.csv",
    index=False
)

pd.Series(lof_features_revised).to_csv(
    output_dir / "lof_features_used.csv",
    index=False,
    header=["feature"]
)

# Copiar resultados a Drive
drive_output = Path(
    "/content/drive/MyDrive/BI-PYMES-ML/results/metrics"
)
drive_output.mkdir(parents=True, exist_ok=True)

for filename in [
    "local_outlier_factor_scores.csv",
    "local_outlier_factor_model.joblib",
    "statistical_baseline_mad_scores.csv",
    "lof_features_used.csv"
]:
    shutil.copy2(
        output_dir / filename,
        drive_output / filename
    )

print("Resultados de Notebook 05 guardados correctamente.")

Resultados de Notebook 05 guardados correctamente.
